# Sesión 2 — RAG: *Retrieval-Augmented Generation*

Notebook de soporte. Asume la Sesión 1 (embeddings y chunking). Cubre:

1. Qué es RAG y por qué se usa.
2. Anatomía: indexación (offline) y recuperación (online).
3. El **índice vectorial** con FAISS (construir, guardar, cargar, buscar).
4. Recuperación base y agregación a nivel documento.
5. **Consulta hipotética (HyDE)** para mejorar la recuperación.
6. **Re-ranking** con un cross-encoder.



## 0. Entorno

In [1]:
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

np.set_printoptions(precision=4, suppress=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('faiss', faiss.__version__, '| device:', DEVICE)

C:\Users\rufra\Documents\GitHub\CODEFEST - 2026\preparacion\venv\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


faiss 1.8.0 | device: cpu


## 1. Qué es RAG

Un modelo generativo (LLM) responde bien sobre lo que vio en su entrenamiento, pero:
no conoce documentos privados o recientes, y puede *alucinar*. **RAG** resuelve esto
separando dos responsabilidades:

- **Recuperar** (retrieval): buscar en una base de conocimiento los fragmentos relevantes
  a la pregunta. Es un problema de *búsqueda*, no de generación.
- **Generar** (generation): un LLM redacta la respuesta **condicionado** a esos fragmentos.

La calidad final está acotada por la recuperación: si no se recupera la evidencia correcta,
el generador no puede inventarla bien. Por eso este notebook se concentra en el *retriever*.

### 1.1 Flujo

```
OFFLINE (una vez):   documentos -> chunking -> encoder -> vectores -> INDICE (FAISS)
ONLINE (por query):  consulta -> encoder -> vector -> buscar top-k -> [re-rank] -> contexto -> LLM
```

El encoder de la consulta debe ser **el mismo** que el del índice.

## 2. Corpus de ejemplo

Corpus pequeño y multilingüe (ES/EN/PT) sobre temas neutros y variados. Cada entrada
simula un chunk ya fragmentado, con su metadata (`doc_id`, `chunk_id`, texto, idioma).

In [2]:
corpus = [
  {'doc_id':'D01','chunk_id':'D01-c0','lang':'en','text':'Photosynthesis lets plants convert sunlight, water and carbon dioxide into sugars, releasing oxygen.'},
  {'doc_id':'D01','chunk_id':'D01-c1','lang':'en','text':'The oxygen produced by plants during the day is a by-product that most animals need to breathe.'},
  {'doc_id':'D02','chunk_id':'D02-c0','lang':'es','text':'La fotosintesis permite a las plantas transformar la luz solar en energia quimica almacenada como azucares.'},
  {'doc_id':'D03','chunk_id':'D03-c0','lang':'en','text':'A solar panel converts sunlight directly into electricity using photovoltaic cells made of silicon.'},
  {'doc_id':'D03','chunk_id':'D03-c1','lang':'en','text':'The efficiency of a solar panel drops as its temperature rises above the rated operating point.'},
  {'doc_id':'D04','chunk_id':'D04-c0','lang':'pt','text':'Um painel solar transforma a luz do sol em eletricidade por meio de celulas fotovoltaicas de silicio.'},
  {'doc_id':'D05','chunk_id':'D05-c0','lang':'en','text':'Wind turbines generate power by converting the kinetic energy of moving air into rotational motion.'},
  {'doc_id':'D06','chunk_id':'D06-c0','lang':'es','text':'Las turbinas eolicas aprovechan el viento para mover un rotor y producir electricidad sin emisiones directas.'},
  {'doc_id':'D07','chunk_id':'D07-c0','lang':'en','text':'The water cycle moves water between oceans, atmosphere and land through evaporation and precipitation.'},
  {'doc_id':'D08','chunk_id':'D08-c0','lang':'en','text':'Lithium-ion batteries store energy chemically and are widely used in phones and electric vehicles.'},
  {'doc_id':'D09','chunk_id':'D09-c0','lang':'pt','text':'As baterias de litio armazenam energia e sao usadas em carros eletricos e dispositivos moveis.'},
  {'doc_id':'D10','chunk_id':'D10-c0','lang':'en','text':'Coffee is prepared by brewing roasted coffee beans with hot water to extract flavour and caffeine.'},
]
texts = [c['text'] for c in corpus]
print(len(corpus), 'chunks |', len(set(c['doc_id'] for c in corpus)), 'documentos')

12 chunks | 10 documentos


## 3. Índice vectorial con FAISS

Pasos: (1) codificar los chunks y **normalizar** a norma unitaria; (2) crear un
`IndexFlatIP` (producto interno). Con vectores normalizados, producto interno = **coseno**.

`IndexFlatIP` es exacto (compara contra todos). Para millones de vectores se usan índices
aproximados (`IndexIVFFlat`, `IndexHNSWFlat`), que cambian algo de exactitud por velocidad.

In [3]:
encoder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)

emb = encoder.encode(texts, normalize_embeddings=True, convert_to_numpy=True).astype('float32')
d = emb.shape[1]
print('matriz de embeddings:', emb.shape, '| dimension d =', d)

index = faiss.IndexFlatIP(d)   # producto interno = coseno (vectores normalizados)
index.add(emb)
print('vectores en el indice:', index.ntotal)

C:\Users\rufra\Documents\GitHub\CODEFEST - 2026\preparacion\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


matriz de embeddings: (12, 384) | dimension d = 384
vectores en el indice: 12


### 3.1 Persistencia

El índice se serializa a disco con la API nativa de FAISS y se recarga sin reindexar.
La metadata va **por separado** (aquí en memoria; en producción, JSONL/SQLite) porque
FAISS solo guarda vectores + un id entero interno.

In [4]:
import os
os.makedirs('data', exist_ok=True)
faiss.write_index(index, 'data/demo.faiss')
reloaded = faiss.read_index('data/demo.faiss')
print('recargado, ntotal =', reloaded.ntotal)

recargado, ntotal = 12


### 3.2 Buscar

La consulta se codifica con el **mismo** encoder, se normaliza y se busca el top-k.
FAISS devuelve las posiciones internas y las puntuaciones; la posición se mapea a la
metadata para recuperar texto y `doc_id`.

In [5]:
def search(query, k=5, index=index):
    q = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, ids = index.search(q, k)
    out = []
    for score, i in zip(scores[0], ids[0]):
        m = corpus[i]
        out.append({'score':float(score),'doc_id':m['doc_id'],'chunk_id':m['chunk_id'],
                    'lang':m['lang'],'text':m['text']})
    return out

query = 'Como generan oxigeno las plantas?'   # consulta en espanol
for r in search(query, k=4):
    print(f"{r['score']:.3f} [{r['lang']}] {r['doc_id']:>4} | {r['text'][:70]}")

0.753 [en]  D01 | The oxygen produced by plants during the day is a by-product that most
0.706 [en]  D01 | Photosynthesis lets plants convert sunlight, water and carbon dioxide 
0.558 [es]  D02 | La fotosintesis permite a las plantas transformar la luz solar en ener
0.388 [en]  D05 | Wind turbines generate power by converting the kinetic energy of movin


La consulta en español recupera chunks en inglés y español sobre fotosíntesis/oxígeno:
el encoder multilingüe alinea los idiomas en el mismo espacio.

### 3.3 Agregación a nivel documento

La búsqueda opera sobre chunks. Para rankear **documentos** se agrupan los chunks por
`doc_id` y se combina su puntuación. Una estrategia simple y robusta es *max pooling*:
la relevancia del documento = la de su mejor chunk.

In [6]:
from collections import defaultdict

def rank_documents(query, k_chunks=10, top_docs=3):
    hits = search(query, k=k_chunks)
    best = defaultdict(float)
    for h in hits:
        best[h['doc_id']] = max(best[h['doc_id']], h['score'])
    ranked = sorted(best.items(), key=lambda x: -x[1])[:top_docs]
    return ranked

print('Documentos para:', query)
for doc_id, score in rank_documents(query):
    print(f'  {doc_id}  score={score:.3f}')

Documentos para: Como generan oxigeno las plantas?
  D01  score=0.753
  D02  score=0.558
  D05  score=0.388


## 4. Consulta hipotética (HyDE)

Problema: una consulta vaga y coloquial ('why do panels work worse when it is hot?') tiene
poca señal de vocabulario tecnico y su vector
queda a medio camino entre varios temas. **HyDE** (*Hypothetical Document Embeddings*):
en vez de codificar la consulta, se codifica una **respuesta hipotética** —un párrafo que
*se parece* a un buen documento de respuesta— y se busca con ese vector, que cae más cerca
de los documentos reales relevantes.

En un sistema completo, esa respuesta hipotética la redacta un LLM. Aquí la escribimos a
mano para aislar y medir el **efecto sobre la recuperación**.

In [7]:
consulta_corta = 'why do panels work worse when it is hot?'   # pregunta vaga, sin terminos tecnicos

# Respuesta hipotetica (en produccion: generada por un LLM a partir de la consulta)
hyde = ('A solar panel loses efficiency when its temperature increases above the '
        'rated operating point, so hot weather reduces its electrical output.')

def top_scores(vec_text, k=3):
    v = encoder.encode([vec_text], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    s, ids = index.search(v, k)
    return [(corpus[i]['chunk_id'], float(sc)) for sc,i in zip(s[0], ids[0])]

print('Base (solo consulta):')
for cid,sc in top_scores(consulta_corta): print(f'   {sc:.3f}  {cid}')
print('HyDE (respuesta hipotetica):')
for cid,sc in top_scores(hyde): print(f'   {sc:.3f}  {cid}')

Base (solo consulta):
   0.576  D03-c1
   0.344  D03-c0
   0.294  D04-c0
HyDE (respuesta hipotetica):
   0.900  D03-c1
   0.555  D03-c0
   0.551  D04-c0


Con HyDE los chunks correctos (paneles + temperatura) suben su puntuación y su posición:
el vector de un párrafo rico se ancla mejor en la región del espacio donde viven las
respuestas reales. Coste: una llamada extra de generación por consulta.

## 5. Re-ranking con cross-encoder

El *retriever* (bi-encoder) codifica consulta y chunk **por separado** y compara vectores:
rápido, escala a millones, pero pierde interacción fina entre las palabras de ambos.

Un **cross-encoder** recibe el par `(consulta, chunk)` **junto** y produce un score de
relevancia. Es mucho más preciso pero costoso, así que no se aplica a todo el corpus:
solo se **re-ordena** el top-k del bi-encoder. Patrón estándar: *retrieve k=50, rerank, top 10*.

In [8]:
reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', device=DEVICE)

query = 'what material are solar cells made of?'
candidatos = search(query, k=8)   # recuperacion amplia con el bi-encoder

pairs = [(query, c['text']) for c in candidatos]
rr_scores = reranker.predict(pairs)
for c, s in zip(candidatos, rr_scores):
    c['rerank'] = float(s)

reordenado = sorted(candidatos, key=lambda c: -c['rerank'])

print('ANTES (bi-encoder)            ->  DESPUES (cross-encoder)')
for a, b in zip(candidatos, reordenado):
    print(f"  {a['chunk_id']:>7} {a['score']:.3f}   |   {b['chunk_id']:>7} rr={b['rerank']:+.2f}")

C:\Users\rufra\Documents\GitHub\CODEFEST - 2026\preparacion\venv\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


ANTES (bi-encoder)            ->  DESPUES (cross-encoder)
   D04-c0 0.667   |    D03-c0 rr=+1.32
   D03-c0 0.639   |    D04-c0 rr=-2.27
   D03-c1 0.407   |    D01-c0 rr=-3.54
   D09-c0 0.393   |    D08-c0 rr=-4.23
   D02-c0 0.368   |    D01-c1 rr=-4.76
   D08-c0 0.334   |    D02-c0 rr=-4.89
   D01-c0 0.331   |    D03-c1 rr=-5.86
   D01-c1 0.200   |    D09-c0 rr=-6.24


El cross-encoder empuja arriba el chunk que responde literalmente la pregunta (paneles +
temperatura), aunque el bi-encoder lo tuviera algo más abajo. El re-ranking corrige el
orden fino sin recodificar todo el índice.

## 6. Pipeline completo

Ensamblado de las piezas: recuperación amplia -> (HyDE opcional) -> re-ranking -> contexto.
El bloque de generación quedaría al final (fuera de foco aquí).

In [9]:
def retrieve(query, k_first=8, k_final=3, use_reranker=True):
    cands = search(query, k=k_first)
    if use_reranker:
        scores = reranker.predict([(query, c['text']) for c in cands])
        for c,s in zip(cands, scores): c['final'] = float(s)
        cands = sorted(cands, key=lambda c: -c['final'])
    else:
        for c in cands: c['final'] = c['score']
    return cands[:k_final]

for r in retrieve('baterias para autos electricos', k_final=3):
    print(f"{r['final']:+.3f} [{r['lang']}] {r['doc_id']} | {r['text'][:70]}")

+3.773 [en] D08 | Lithium-ion batteries store energy chemically and are widely used in p
+3.435 [pt] D09 | As baterias de litio armazenam energia e sao usadas em carros eletrico
-5.011 [en] D05 | Wind turbines generate power by converting the kinetic energy of movin


## 7. Resumen

- RAG = **recuperar** evidencia + **generar** condicionado a ella; la recuperación es el techo.
- FAISS indexa vectores normalizados con `IndexFlatIP` (= coseno); se persiste aparte de la metadata.
- Se busca a nivel chunk y se **agrega** a nivel documento (p. ej. max pooling).
- **HyDE**: buscar con una respuesta hipotética acerca el vector a los documentos correctos.
- **Re-ranking** con cross-encoder: reordenar el top-k del bi-encoder para precisión fina.